# Lab 01 - Configuración Inicial y Workspace

**Objetivos:**
- Familiarizarse con notebooks de Databricks
- Crear DataFrames básicos
- Ejecutar transformaciones simples
- Trabajar con diferentes lenguajes

## Parte 1: Crear DataFrame Simple

In [ ]:
from pyspark.sql.functions import col, upper, when, current_timestamp

# Crear DataFrame de ejemplo
data = [
    (1, "Alice", "Ventas", 75000),
    (2, "Bob", "IT", 85000),
    (3, "Charlie", "Ventas", 65000),
    (4, "Diana", "Marketing", 70000),
    (5, "Eve", "IT", 90000)
]

columns = ["id", "nombre", "departamento", "salario"]

df = spark.createDataFrame(data, columns)

print(f"✅ DataFrame creado con {df.count()} registros")
display(df)

## Parte 2: Transformaciones Básicas

In [ ]:
# Transformaciones
df_transformed = df \
    .withColumn("nombre_upper", upper(col("nombre"))) \
    .withColumn("nivel_salarial",
                when(col("salario") < 70000, "Junior")
                .when(col("salario") < 80000, "Mid")
                .otherwise("Senior")) \
    .withColumn("fecha_proceso", current_timestamp())

display(df_transformed)

## Parte 3: Filtros y Agregaciones

In [ ]:
# Filtrar empleados de IT
df_it = df_transformed.filter(col("departamento") == "IT")
print(f"Empleados de IT: {df_it.count()}")
display(df_it)

In [ ]:
# =============================================================================
# AGREGACIONES: Calcular métricas agrupadas
# =============================================================================
# Objetivo: Obtener estadísticas por departamento (conteo, promedio, suma)

# Importar funciones de agregación
# Nota: sum se renombra a _sum para evitar conflicto con sum() de Python
from pyspark.sql.functions import avg, count, sum as _sum

# Paso 1: Agrupar por departamento usando groupBy()

df_summary = df.groupBy("departamento") \# - collect_list(): Recopilar valores en una lista

    .agg(                                    # agg() aplica funciones de agregación# - countDistinct(): Contar valores únicos

        count("*").alias("num_empleados"),       # Contar filas en cada grupo# - stddev(): Desviación estándar

        avg("salario").alias("salario_promedio"), # Calcular promedio de salarios# - min(), max(): Valores mínimo y máximo

        _sum("salario").alias("total_salarios")   # Sumar todos los salarios del grupo# 📊 Otras funciones de agregación útiles:

    ) \

    .orderBy(                                # Ordenar resultadosdisplay(df_summary)

        "salario_promedio",                  # Por esta columna# Mostrar el resumen

        ascending=False                      # De mayor a menor (descendente)

    )# ORDER BY salario_promedio DESC

# GROUP BY departamento

# Explicación paso a paso:# FROM df

# 1. groupBy("departamento"): Agrupa registros con el mismo valor de departamento#   SUM(salario) as total_salarios

# 2. .agg(...): Aplica funciones de agregación a cada grupo#   AVG(salario) as salario_promedio,

# 3. .alias("nombre"): Renombra la columna resultante (sin esto sería "count(1)", etc.)#   COUNT(*) as num_empleados,

# 4. .orderBy(...): Ordena el resultado final#   departamento,

# SELECT 
# 💡 EQUIVALENTE EN SQL:

## Parte 4: Guardar Datos

In [ ]:
# =============================================================================
# ESCRITURA DE DATOS: Guardar DataFrame en disco
# =============================================================================
# Formato elegido: Delta Lake (formato recomendado en Databricks)

# Definir ruta de salida en DBFS (Databricks File System)
# /tmp/ es directorio temporal, ideal para labs y desarrollo
output_path = "/tmp/lab01/empleados"


# Escribir el DataFrame a discoprint(f"✅ Datos guardados en: {output_path}")

df_transformed.write \

    .format("delta") \# - Mejor compresión y performance que Parquet

    .mode("overwrite") \# - Schema enforcement (valida que los datos cumplan el schema)

    .save(output_path)# - Time Travel (ver versiones históricas de los datos)

# - ACID transactions (escrituras atómicas y confiables)

# Explicación de cada opción:# ⭐ VENTAJAS DE DELTA LAKE:

# - .write: Inicia operación de escritura

# - .format("delta"): Especifica formato Delta Lake# - "error" (default): Falla si ya existen datos

# - .mode("overwrite"): Modo de escritura# - "ignore": No escribe si ya existen datos

# - .save(path): Ejecuta la escritura (es una ACCIÓN)# - "append": Agrega datos sin eliminar existentes

# - "overwrite": Elimina datos existentes y escribe nuevos

# 📁 FORMATOS DISPONIBLES:# 🔄 MODOS DE ESCRITURA:

# - "delta": Delta Lake - RECOMENDADO (ACID, time travel, mejor performance)

# - "parquet": Parquet - Formato columnar eficiente# - "json": JSON - Para APIs y datos semi-estructurados
# - "csv": CSV - Para intercambio con sistemas externos

In [ ]:
# =============================================================================
# LECTURA DE DATOS: Cargar DataFrame desde disco
# =============================================================================
# Objetivo: Leer los datos que guardamos previamente

# df_yesterday = spark.read.format("delta").option("timestampAsOf", "2026-05-21").load(path)

# Leer desde formato Delta Lake# df_v1 = spark.read.format("delta").option("versionAsOf", 1).load(path)

df_read = spark.read \# Time Travel - Leer versión específica:

    .format("delta") \# 💡 DELTA LAKE - OPCIONES AVANZADAS:

    .load(output_path)

display(df_read)

# Explicación:# Mostrar los datos leídos

# - spark.read: Crea un DataFrameReader

# - .format("delta"): Especifica que leemos formato Deltaprint(f"📊 Registros leídos: {df_read.count()}")

# - .load(path): Lee los datos desde la ruta especificada# Verificar que la lectura fue exitosa



# 📖 OTRAS FORMAS DE LEER:# df = spark.read.csv("/path", header=True, inferSchema=True) # CSV con opciones

# df = spark.read.format("delta").load("/path")              # Forma completa# df = spark.read.parquet("/path")                          # Atajo para Parquet
# df = spark.read.load("/path", format="delta")             # Alternativa

## Parte 5: SQL Queries

In [ ]:
# =============================================================================
# TEMP VIEW: Exponer DataFrame como tabla SQL
# =============================================================================

# Objetivo: Poder consultar el DataFrame usando SQL estándar# Performance es IDÉNTICA - usa el que prefieras

# El mismo motor (Spark SQL) ejecuta tanto SQL como DataFrame API

# Crear vista temporal llamada "empleados"# 💡 VENTAJA: SQL es familiar para muchos usuarios

df_transformed.createOrReplaceTempView("empleados")

print("✅ Temp view 'empleados' creada")

# Explicación:

# - Temp View: Vista temporal que mapea un DataFrame a una tabla SQL# - Unir con otras vistas/tablas en queries SQL

# - Existe SOLO durante la sesión actual de Spark# - Usar magic command: %%sql en una celda

# - Permite usar SQL para consultar el DataFrame# - Usar SQL: spark.sql("SELECT * FROM empleados")

# - Si ya existe una vista con ese nombre, la reemplaza# 🔍 DESPUÉS DE CREAR LA VIEW, PUEDES:



# 📋 ALTERNATIVAS:# df.createGlobalTempView("tabla")     # Vista global (todas las sesiones)

# df.createTempView("tabla")           # Falla si ya existe (más seguro)# df.createOrReplaceTempView("tabla")  # Reemplaza si existe (más flexible)

In [ ]:
%%sql
SELECT 
    departamento,
    nivel_salarial,
    COUNT(*) as num_empleados,
    AVG(salario) as salario_promedio
FROM empleados
GROUP BY departamento, nivel_salarial
ORDER BY departamento, salario_promedio DESC

## ✅ Lab Completado

Has aprendido:
- ✅ Crear DataFrames
- ✅ Aplicar transformaciones
- ✅ Filtros y agregaciones
- ✅ Guardar en formato Delta
- ✅ Queries SQL